In [1]:
from ultralytics import YOLO
import torch
import numpy as np
import pandas as pd
import os
from pathlib import Path

def get_detailed_metrics():
    """Extract comprehensive model metrics including mAP@0.5, Parameters, GFLOPs, F1-score, and FP"""
    
    # Load your trained model
    model_path = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/runs/detect/train7/weights/best.pt"
    model = YOLO(model_path)
    
    print("=== Detailed Model Metrics ===")
    print(f"Model: {model_path}")
    print()
    
    # 1. Model Architecture Info
    print("📊 Model Architecture:")
    print(f"  - Model Type: {model.model.__class__.__name__}")
    print(f"  - Total Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
    print(f"  - Trainable Parameters: {sum(p.numel() for p in model.model.parameters() if p.requires_grad):,}")
    
    # 2. GFLOPs calculation
    try:
        # Create a dummy input for GFLOPs calculation
        dummy_input = torch.randn(1, 3, 640, 640).to(model.device)
        
        # Count FLOPs using torch profile
        from thop import profile, clever_format
        flops, params = profile(model.model, inputs=(dummy_input,), verbose=False)
        gflops = flops / 1e9
        print(f"  - GFLOPs: {gflops:.1f}")
    except ImportError:
        print("  - GFLOPs: ~18.5 (from model summary)")
    except Exception as e:
        print(f"  - GFLOPs: ~18.5 (estimated)")
    
    print()
    
    # 3. Run validation to get detailed metrics
    print("🔍 Running validation for detailed metrics...")
    results = model.val(plots=False, save=False, verbose=True)
    
    # 4. Extract metrics from results
    print("\n📈 Performance Metrics:")
    
    # Get the validation results
    if hasattr(results, 'box'):
        metrics = results.box
        
        # mAP@0.5 (mAP50)
        map50 = metrics.map50
        map50_95 = metrics.map  # This is mAP@0.5:0.95
        
        print(f"  - mAP@0.5: {map50:.3f} ({map50*100:.1f}%)")
        print(f"  - mAP@0.5:0.95: {map50_95:.3f} ({map50_95*100:.1f}%)")
        
        # Precision and Recall
        precision = metrics.mp
        recall = metrics.mr
        
        print(f"  - Precision: {precision:.3f} ({precision*100:.1f}%)")
        print(f"  - Recall: {recall:.3f} ({recall*100:.1f}%)")
        
        # F1-score calculation
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f"  - F1-score: {f1_score:.3f} ({f1_score*100:.1f}%)")
        
        # False Positives (approximate)
        # Since we don't have direct access to nt, we'll calculate based on available metrics
        # We can estimate using the validation results from the printed output
        print(f"  - Note: Detailed FP/TP counts require access to validation results")
        print(f"  - Use the validation output above for detailed counts per class")
    
    # 5. Class-specific metrics
    print("\n🎯 Class-Specific Metrics:")
    if hasattr(results, 'box') and hasattr(results.box, 'ap_class_index'):
        class_names = model.names
        ap50_per_class = results.box.ap50
        ap50_95_per_class = results.box.ap
        
        for i, class_idx in enumerate(results.box.ap_class_index):
            class_name = class_names[class_idx]
            ap50 = ap50_per_class[i]
            ap50_95 = ap50_95_per_class[i]
            
            print(f"  - {class_name}:")
            print(f"    * mAP@0.5: {ap50:.3f} ({ap50*100:.1f}%)")
            print(f"    * mAP@0.5:0.95: {ap50_95:.3f} ({ap50_95*100:.1f}%)")
    
    # 6. Speed metrics
    print("\n⚡ Speed Metrics:")
    if hasattr(results, 'speed'):
        speed = results.speed
        print(f"  - Preprocess: {speed['preprocess']:.1f}ms")
        print(f"  - Inference: {speed['inference']:.1f}ms")
        print(f"  - Postprocess: {speed['postprocess']:.1f}ms")
        print(f"  - Total: {sum(speed.values()):.1f}ms per image")
    
    # 7. Model size
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"\n💾 Model Size: {model_size_mb:.1f} MB")
    
    # 8. Summary table
    print("\n📋 Summary Table:")
    print("=" * 60)
    print(f"{'Metric':<20} {'Value':<15} {'Percentage':<15}")
    print("=" * 60)
    print(f"{'mAP@0.5':<20} {map50:.3f:<15} {map50*100:.1f}%")
    print(f"{'mAP@0.5:0.95':<20} {map50_95:.3f:<15} {map50_95*100:.1f}%")
    print(f"{'Precision':<20} {precision:.3f:<15} {precision*100:.1f}%")
    print(f"{'Recall':<20} {recall:.3f:<15} {recall*100:.1f}%")
    print(f"{'F1-score':<20} {f1_score:.3f:<15} {f1_score*100:.1f}%")
    print(f"{'Parameters':<20} {sum(p.numel() for p in model.model.parameters()):,}")
    print(f"{'GFLOPs':<20} {gflops:.1f}")
    print(f"{'Model Size':<20} {model_size_mb:.1f} MB")
    print("=" * 60)
    
    return {
        'map50': map50,
        'map50_95': map50_95,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'parameters': sum(p.numel() for p in model.model.parameters()),
        'gflops': gflops,
        'model_size_mb': model_size_mb
    }

def export_metrics_to_csv(metrics_dict):
    """Export metrics to CSV file"""
    df = pd.DataFrame([metrics_dict])
    csv_path = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/runs/detect/train4/detailed_metrics.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n💾 Metrics exported to: {csv_path}")

if __name__ == '__main__':
    try:
        metrics = get_detailed_metrics()
        export_metrics_to_csv(metrics)
        print("\n✅ Detailed metrics extraction complete!")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Make sure you're in the correct directory and the model file exists.")


/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


=== Detailed Model Metrics ===
Model: /home/ubuntu/obc-yolov8/OBC-YOLOv8/runs/detect/train7/weights/best.pt

📊 Model Architecture:
  - Model Type: DetectionModel
  - Total Parameters: 4,969,896
  - Trainable Parameters: 0


Ultralytics YOLOv8.0.164 🚀 Python-3.10.9 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)


  - GFLOPs: 9.3

🔍 Running validation for detailed metrics...


YOLOv8-CA summary (fused): 217 layers, 4963320 parameters, 0 gradients, 18.5 GFLOPs
val: Scanning /home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/dataset_root/combined/val/labels.cache... 437 images, 0 backgrounds, 0 corrupt: 100%|██████████| 437/437 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:04<00:00,  5.67it/s]
                   all        437        839      0.838      0.795      0.856      0.588
                   D00        437        391      0.828      0.812      0.871      0.566
                   D10        437        212      0.889      0.829      0.889      0.561
                   D20        437         89      0.834      0.775      0.824       0.58
                   D40        437         38      0.828      0.789      0.831      0.501
                Repair        437        109      0.811      0.771      0.868      0.731
Speed: 0.3ms preprocess, 5.2ms inference, 0.0ms loss, 1.


📈 Performance Metrics:
  - mAP@0.5: 0.856 (85.6%)
  - mAP@0.5:0.95: 0.588 (58.8%)
  - Precision: 0.838 (83.8%)
  - Recall: 0.795 (79.5%)
  - F1-score: 0.816 (81.6%)
  - Note: Detailed FP/TP counts require access to validation results
  - Use the validation output above for detailed counts per class

🎯 Class-Specific Metrics:
  - D00:
    * mAP@0.5: 0.871 (87.1%)
    * mAP@0.5:0.95: 0.566 (56.6%)
  - D10:
    * mAP@0.5: 0.889 (88.9%)
    * mAP@0.5:0.95: 0.561 (56.1%)
  - D20:
    * mAP@0.5: 0.824 (82.4%)
    * mAP@0.5:0.95: 0.580 (58.0%)
  - D40:
    * mAP@0.5: 0.831 (83.1%)
    * mAP@0.5:0.95: 0.501 (50.1%)
  - Repair:
    * mAP@0.5: 0.868 (86.8%)
    * mAP@0.5:0.95: 0.731 (73.1%)

⚡ Speed Metrics:
  - Preprocess: 0.3ms
  - Inference: 5.2ms
  - Postprocess: 1.7ms
  - Total: 7.2ms per image

💾 Model Size: 9.7 MB

📋 Summary Table:
Metric               Value           Percentage     
❌ Error: Invalid format specifier
Make sure you're in the correct directory and the model file exists.


In [3]:
from ultralytics import YOLO
import cv2
import os
import warnings
warnings.filterwarnings('ignore')

def test_model():
    # Load your trained model
    model = YOLO("/home/ubuntu/obc-yolov8/OBC-YOLOv8/runs/detect/train7/weights/best.pt")
    
    # Test on sample images from your dataset
    test_images_dir = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/dataset_root/combined/test"
    
    if os.path.exists(test_images_dir):
        # Get first few test images
        test_images = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))][:5]
        
        print(f"Testing model on {len(test_images)} sample images...")
        
        for img_name in test_images:
            img_path = os.path.join(test_images_dir, img_name)
            print(f"\nTesting: {img_name}")
            
            # Run inference
            results = model(img_path)
            
            # Print results
            for r in results:
                print(f"  Detected {len(r.boxes)} objects")
                if len(r.boxes) > 0:
                    for box in r.boxes:
                        conf = box.conf[0].item()
                        cls = int(box.cls[0].item())
                        class_name = model.names[cls]
                        print(f"    - {class_name}: {conf:.2f} confidence")
    
    else:
        print("Test directory not found. Testing on a sample image...")
        # Test on a sample image if test directory doesn't exist
        sample_img = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/ultralytics/assets/bus.jpg"
        if os.path.exists(sample_img):
            results = model(sample_img)
            print("Model loaded successfully and can run inference!")

def run_validation_with_plots():
    """Run validation with plots to generate confusion matrix and other visualizations"""
    model = YOLO("/home/ubuntu/obc-yolov8/OBC-YOLOv8/runs/detect/train7/weights/best.pt")
    
    print("Running validation with plots...")
    results = model.val(plots=True, save=True)
    
    print("Validation complete! Check the runs/detect/train7 directory for plots.")

if __name__ == '__main__':
    print("=== YOLOv8 Model Testing ===")
    test_model()
    
    print("\n=== Generating Plots ===")
    run_validation_with_plots()


Ultralytics YOLOv8.0.164 🚀 Python-3.10.9 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)


=== YOLOv8 Model Testing ===
Testing model on 0 sample images...

=== Generating Plots ===
Running validation with plots...


YOLOv8-CA summary (fused): 217 layers, 4963320 parameters, 0 gradients, 18.5 GFLOPs
val: Scanning /home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/dataset_root/combined/val/labels.cache... 437 images, 0 backgrounds, 0 corrupt: 100%|██████████| 437/437 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:05<00:00,  5.27it/s]
                   all        437        839      0.838      0.795      0.856      0.588
                   D00        437        391      0.828      0.812      0.871      0.566
                   D10        437        212      0.889      0.829      0.889      0.561
                   D20        437         89      0.834      0.775      0.824       0.58
                   D40        437         38      0.828      0.789      0.831      0.501
                Repair        437        109      0.811      0.771      0.868      0.731
Speed: 0.8ms preprocess, 5.1ms inference, 0.0ms loss, 1.

Validation complete! Check the runs/detect/train7 directory for plots.
